<a href="https://colab.research.google.com/github/JeffersonRodrigues9/Automacao_com_python/blob/main/Deteccao_Fraude_Documentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Desenvolvi este projeto em Python para automatizar a extração e análise de documentos PDF
# em grandes volumes de arquivos. O objetivo do script é identificar possíveis duplicidades,
# inconsistências e padrões suspeitos, realizando a leitura automática de informações
# como CNPJ, CPF, valores, datas, nomes e validação de assinaturas, gerando relatórios
# estruturados em Excel para apoio em processos operacionais, financeiros e validações documentais.

!pip install pdfplumber pandas tqdm openpyxl -q

import os
import re
import time
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import pdfplumber
import pandas as pd
from tqdm.notebook import tqdm
from google.colab import files, drive
from openpyxl.styles import PatternFill, Font, Alignment

USE_DRIVE = True
PASTA_PDFS = ""
ARQUIVO_SAIDA = ".xlsx"
NUM_THREADS = 4
LOTE_SALVAMENTO = 1000

if USE_DRIVE:
    drive.mount("")
    print(f"Google Drive montado!")
else:
    print("Fazendo upload dos PDFs...")
    uploaded = files.upload()
    PASTA_PDFS = ""
    os.makedirs(PASTA_PDFS, exist_ok=True)

    for nome, conteudo in uploaded.items():
        with open(f"{PASTA_PDFS}/{nome}", "wb") as f:
            f.write(conteudo)

    print(f"{len(uploaded)} arquivo(s) carregado(s)!")

PADROES = {
    "cnpj": [r"\d{2}[\.\-]?\d{3}[\.\-]?\d{3}[\/\\]?\d{4}[\-\.]?\d{2}"],

    "cpf": [r"\d{3}[\.\-]?\d{3}[\.\-]?\d{3}[\-\.]?\d{2}"],

    "data_emissao": [
        r"(?:data\s*(?:de\s*)?emiss[aã]o|emitid[oa]\s*em|data)[:\s]*(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})",
        r"(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{4})",
        r"(\d{4}[\/\-\.]\d{1,2}[\/\-\.]\d{1,2})",
    ],

    "valor": [
        r"(?:valor\s*total|total\s*(?:a\s*pagar|do\s*documento|geral)|valor)[:\s]*R?\$?\s*([\d\.]+[\,\.][\d]{2})",
        r"R\$\s*([\d\.]+[\,\.][\d]{2})",
        r"(?:total)[:\s]*\s*([\d\.]+[\,\.][\d]{2})",
    ],

    "nome_empresa": [
        r"(?:raz[aã]o\s*social|empresa|contratad[ao]|prestador)[:\s]*([A-ZÁÀÂÃÉÊÍÓÔÕÚÇ][A-Za-záàâãéêíóôõúçÁÀÂÃÉÊÍÓÔÕÚÇ\s\.\,\-]{5,80}(?:LTDA|S\.?A\.?|ME|EPP|EIRELI|SS|SLU|SA)?)",
        r"([A-ZÁÀÂÃÉÊÍÓÔÕÚÇ][A-Z\s\.]{5,60}(?:LTDA|S\.A\.|ME|EPP|EIRELI|SS|SLU)\.?)",
    ],

    "nome_pessoa": [
        r"(?:nome\s*(?:completo)?|contratante|tomador|cliente|respons[aá]vel|benefici[aá]rio)[:\s]*([A-ZÁÀÂÃÉÊÍÓÔÕÚÇ][a-záàâãéêíóôõúç]+(?:\s+[A-ZÁÀÂÃÉÊÍÓÔÕÚÇ][a-záàâãéêíóôõúç]+){1,5})",
        r"(?:Sr\.?|Sra\.?|Dr\.?|Dra\.?)\s*([A-ZÁÀÂÃÉÊÍÓÔÕÚÇ][a-záàâãéêíóôõúç]+(?:\s+[A-ZÁÀÂÃÉÊÍÓÔÕÚÇ][a-záàâãéêíóôõúç]+){1,4})",
    ],

    "assinatura": [
        r"(?:assinado|assinatura|subscri|rubric|ass\.?\s*:)",
        r"(?:certificado\s*digital|assina(?:do)?\s*digitalmente|ICP[\-\s]Brasil)",
        r"(?:assinou|assinante|sign(?:ed|ature))",
    ],
}

def extrair_primeiro(texto, padroes, grupo=1):
    for padrao in padroes:
        try:
            match = re.search(padrao, texto, re.IGNORECASE | re.MULTILINE)

            if match:
                return match.group(grupo).strip() if grupo else match.group(0).strip()

        except Exception:
            continue

    return ""

def extrair_todos_cnpjs(texto):
    encontrados = []

    for p in PADROES["cnpj"]:
        encontrados += re.findall(p, texto)

    vistos, unicos = set(), []

    for c in encontrados:
        limpo = re.sub(r"[^\d]", "", c)

        if limpo not in vistos and len(limpo) == 14:
            vistos.add(limpo)
            unicos.append(c)

    return unicos

def normalizar_valor(valor_str):
    if not valor_str:
        return None

    try:
        v = valor_str.strip()

        if "." in v and "," in v:
            v = v.replace(".", "").replace(",", ".")
        elif "," in v:
            v = v.replace(",", ".")

        return round(float(re.sub(r"[^\d\.]", "", v)), 2)

    except Exception:
        return None

def normalizar_data(data_str):
    if not data_str:
        return ""

    for fmt in (
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%d.%m.%Y",
        "%Y/%m/%d",
        "%Y-%m-%d",
        "%d/%m/%y"
    ):
        try:
            return datetime.strptime(data_str.strip(), fmt).strftime("%d/%m/%Y")

        except ValueError:
            continue

    return data_str.strip()

def detectar_assinatura(texto):
    for padrao in PADROES["assinatura"]:
        if re.search(padrao, texto, re.IGNORECASE):
            return "SIM"

    return "NÃO"

def processar_pdf(caminho_pdf):
    resultado = {
        "arquivo": os.path.basename(caminho_pdf),
        "caminho": caminho_pdf,
        "cnpj": "",
        "cpf": "",
        "nome_empresa": "",
        "nome_pessoa": "",
        "data_emissao": "",
        "valor_total": None,
        "assinado": "NÃO",
        "paginas": 0,
        "status": "OK",
        "erro": "",
    }

    try:
        with pdfplumber.open(caminho_pdf) as pdf:
            resultado["paginas"] = len(pdf.pages)
            texto = "\n".join(p.extract_text() or "" for p in pdf.pages)

        if not texto.strip():
            resultado["status"] = "SEM_TEXTO"
            return resultado

        cnpjs = extrair_todos_cnpjs(texto)
        resultado["cnpj"] = " | ".join(cnpjs[:3])

        cpf_raw = extrair_primeiro(texto, PADROES["cpf"], grupo=0)

        resultado["cpf"] = (
            cpf_raw if len(re.sub(r"[^\d]", "", cpf_raw)) == 11 else ""
        )

        resultado["nome_empresa"] = extrair_primeiro(
            texto,
            PADROES["nome_empresa"]
        )

        resultado["nome_pessoa"] = extrair_primeiro(
            texto,
            PADROES["nome_pessoa"]
        )

        resultado["data_emissao"] = normalizar_data(
            extrair_primeiro(texto, PADROES["data_emissao"])
        )

        resultado["valor_total"] = normalizar_valor(
            extrair_primeiro(texto, PADROES["valor"])
        )

        resultado["assinado"] = detectar_assinatura(texto)

    except Exception as e:
        resultado["status"] = "ERRO"
        resultado["erro"] = str(e)[:200]

    return resultado

def salvar_excel(registros, caminho):
    df = pd.DataFrame(registros)

    colunas = [
        "arquivo",
        "cnpj",
        "cpf",
        "nome_empresa",
        "nome_pessoa",
        "data_emissao",
        "valor_total",
        "assinado",
        "paginas",
        "status",
        "erro",
        "caminho"
    ]

    df = df[[c for c in colunas if c in df.columns]]

    df.columns = [
        "Arquivo",
        "CNPJ",
        "CPF",
        "Nome Empresa",
        "Nome Pessoa",
        "Data Emissão",
        "Valor Total (R$)",
        "Assinado",
        "Páginas",
        "Status",
        "Erro",
        "Caminho"
    ]

    with pd.ExcelWriter(caminho, engine="openpyxl") as writer:
        df.to_excel(
            writer,
            index=False,
            sheet_name="Extração Completa"
        )

        ws = writer.sheets["Extração Completa"]

        header_fill = PatternFill("solid", fgColor="1F3864")
        header_font = Font(color="FFFFFF", bold=True, size=11)

        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center")

        for i, w in enumerate(
            [30,20,16,40,35,14,16,10,8,10,30,60],
            1
        ):
            ws.column_dimensions[
                ws.cell(1, i).column_letter
            ].width = w

        fill_par  = PatternFill("solid", fgColor="EBF0FA")
        fill_imp  = PatternFill("solid", fgColor="FFFFFF")
        fill_erro = PatternFill("solid", fgColor="FFE0E0")
        fill_sem  = PatternFill("solid", fgColor="FFF3CD")

        for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
            status = row[9].value if len(row) > 9 else ""

            fill = (
                fill_erro if status == "ERRO"
                else fill_sem if status == "SEM_TEXTO"
                else fill_par if row[0].row % 2 == 0
                else fill_imp
            )

            for cell in row:
                cell.fill = fill

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        resumo = pd.DataFrame({
            "Métrica": [
                "Total de PDFs",
                "Sucesso",
                "Sem texto",
                "Com erro",
                "Com CNPJ",
                "Com CPF",
                "Com valor",
                "Com data",
                "Assinados",
                "Valor total (R$)",
                "Gerado em"
            ],

            "Resultado": [
                len(df),
                (df["Status"] == "OK").sum(),
                (df["Status"] == "SEM_TEXTO").sum(),
                (df["Status"] == "ERRO").sum(),
                (df["CNPJ"] != "").sum(),
                (df["CPF"] != "").sum(),
                df["Valor Total (R$)"].notna().sum(),
                (df["Data Emissão"] != "").sum(),
                (df["Assinado"] == "SIM").sum(),
                f"R$ {df['Valor Total (R$)'].sum():,.2f}",
                datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
            ]
        })

        resumo.to_excel(
            writer,
            index=False,
            sheet_name="Resumo"
        )

    print(f"Excel salvo: {caminho}")

pasta = Path(PASTA_PDFS)
arquivos = sorted(pasta.rglob("*.pdf"))
total = len(arquivos)

print(f"{total:,} PDFs encontrados")
print(f"{NUM_THREADS} threads paralelas")
print(f"Salvamento parcial a cada {LOTE_SALVAMENTO:,} PDFs\n")

registros = []
erros = 0
lote_num = 1
t_inicio = time.time()

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futuros = {
        executor.submit(processar_pdf, str(f)): f
        for f in arquivos
    }

    with tqdm(
        total=total,
        desc="Extraindo PDFs",
        unit="PDF",
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
    ) as barra:

        for futuro in as_completed(futuros):

            try:
                resultado = futuro.result()

            except Exception as e:
                resultado = {
                    "arquivo": futuros[futuro].name,
                    "caminho": str(futuros[futuro]),
                    "status": "ERRO",
                    "erro": str(e)[:200],
                }

            registros.append(resultado)

            if resultado.get("status") == "ERRO":
                erros += 1

            if len(registros) % LOTE_SALVAMENTO == 0:
                parcial = f"/content/parcial_lote_{lote_num}.xlsx"

                salvar_excel(registros, parcial)

                tqdm.write(
                    f"Lote {lote_num} salvo -> "
                    f"{len(registros):,} PDFs processados"
                )

                lote_num += 1

            barra.update(1)

print("\nSalvando resultado final...")
salvar_excel(registros, ARQUIVO_SAIDA)

t_total = time.time() - t_inicio

ok = sum(
    1 for r in registros
    if r.get("status") == "OK"
)

sem_txt = sum(
    1 for r in registros
    if r.get("status") == "SEM_TEXTO"
)

print(f"""
{'='*50}
   EXTRAÇÃO CONCLUÍDA
{'='*50}
  PDFs processados : {total:,}
  Sucesso          : {ok:,}
  Sem texto        : {sem_txt:,}
  Com erro         : {erros:,}
  Tempo total      : {t_total/60:.1f} min
  Velocidade média : {total/t_total:.1f} PDFs/s
{'='*50}
""")

files.download(ARQUIVO_SAIDA)